In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Perceptron

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, SimpleRNN, Embedding, LSTM, GRU

In [4]:
df = pd.read_csv('/content/Womens Clothing E-Commerce Reviews.csv')

In [6]:
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [7]:
df.shape

(23486, 11)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               23486 non-null  int64 
 1   Clothing ID              23486 non-null  int64 
 2   Age                      23486 non-null  int64 
 3   Title                    19676 non-null  object
 4   Review Text              22641 non-null  object
 5   Rating                   23486 non-null  int64 
 6   Recommended IND          23486 non-null  int64 
 7   Positive Feedback Count  23486 non-null  int64 
 8   Division Name            23472 non-null  object
 9   Department Name          23472 non-null  object
 10  Class Name               23472 non-null  object
dtypes: int64(6), object(5)
memory usage: 2.0+ MB


In [14]:
df.isnull().sum()

,0
Review Text,0
Recommended IND,0


In [12]:
df=df[['Review Text','Recommended IND']]

In [13]:
df['Review Text']= df['Review Text'].fillna('')

/tmp/ipykernel_2139/1490730416.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Review Text']= df['Review Text'].fillna('')


In [15]:
X = df['Review Text']
y = df['Recommended IND']

In [16]:
y

,Recommended IND
0,1
1,1
2,0
3,1
4,1
...,...
23481,1
23482,1
23483,0
23484,1


In [17]:
y.value_counts()

,count
Recommended IND,
1,19314
0,4172


In [18]:
X

,Review Text
0,Absolutely wonderful - silky and sexy and comf...
1,Love this dress! it's sooo pretty. i happene...
2,I had such high hopes for this dress and reall...
3,"I love, love, love this jumpsuit. it's fun, fl..."
4,This shirt is very flattering to all due to th...
...,...
23481,I was very happy to snag this dress at such a ...
23482,"It reminds me of maternity clothes. soft, stre..."
23483,"This fit well, but the top was very see throug..."
23484,I bought this dress for a wedding i have this ...


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [21]:
X_train.shape

(18788,)

In [22]:
len(X_train[0])

53

In [24]:
len(X_train[10])

336

In [28]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [32]:
max_length=50
X_train= tf.keras.preprocessing.sequence.pad_sequences(
    X_train_seq,maxlen= max_length,padding='post',truncating='post',
)

X_test= tf.keras.preprocessing.sequence.pad_sequences(
    X_test_seq,maxlen= max_length,padding='post',truncating='post',
)

In [33]:
X_train.shape

(18788, 50)

In [34]:
X_test.shape

(4698, 50)

In [35]:
model= Sequential([
    Input(shape=(max_length,)),
    Embedding(input_dim=vocab_size, output_dim=32),
    #SimpleRNN(64, activation='tanh'),
    LSTM(32, activation='tanh'),
    #GRU(64, activation='tanh'),
    Dense(1,activation='sigmoid')
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 32)         │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 168,353 (657.63 KB)

 Trainable params: 168,353 (657.63 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 32)         │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 168,353 (657.63 KB)

 Trainable params: 168,353 (657.63 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
history=model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_split=0.2,
    batch_size=10,
    verbose=1
)
model.summary()

Epoch 1/5
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 33s 21ms/step - accuracy: 0.8478 - loss: 0.3611 - val_accuracy: 0.8752 - val_loss: 0.3055
Epoch 2/5
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 31s 20ms/step - accuracy: 0.8919 - loss: 0.2622 - val_accuracy: 0.8781 - val_loss: 0.3015
Epoch 3/5
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 42s 21ms/step - accuracy: 0.9132 - loss: 0.2241 - val_accuracy: 0.8803 - val_loss: 0.2927
Epoch 4/5
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 43s 22ms/step - accuracy: 0.9255 - loss: 0.1945 - val_accuracy: 0.8803 - val_loss: 0.3138
Epoch 5/5
1503/1503 ━━━━━━━━━━━━━━━━━━━━ 31s 21ms/step - accuracy: 0.9395 - loss: 0.1658 - val_accuracy: 0.8747 - val_loss: 0.4018


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 32)         │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 505,061 (1.93 MB)

 Trainable params: 168,353 (657.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 336,708 (1.28 MB)

In [38]:
loss,accuracy=model.evaluate(
    X_test,
    y_test

)
print("acc=",accuracy)

147/147 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8712 - loss: 0.4096
acc= 0.8712217807769775
